# 00 — Setup: Metadata Tables & Configuration

**Run ONCE** per environment to create metadata/result tables and register file pairs.
Re-run to update registration (existing entry is replaced).

### Widgets
| # | Widget | Purpose |
|---|--------|---------|
| 1 | `sap_file_path` | Full Volume/DBFS path to SAP source Excel |
| 2 | `databricks_file_path` | Full Volume/DBFS path to Databricks target Excel |
| 3 | `sap_sheet_name` | Sheet name in SAP source file |
| 4 | `databricks_sheet_name` | Sheet name in Databricks target file |
| 5 | `stream_name` | Unique ID for this validation pair |
| 6 | `sap_table_name` | SAP table name (for schema fetcher) |
| 7 | `primary_key_columns` | Comma-separated PK column names |

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  WIDGETS  (7 user-facing — metadata/results DB)
# ══════════════════════════════════════════════════════════════════

VOLUME_PATH = "/Volumes/training/default/validation_volume"

dbutils.widgets.text("sap_file_path",          f"{VOLUME_PATH}/zlofoa01_source_sap.XLSX",        "1. SAP File Path")
dbutils.widgets.text("databricks_file_path",   f"{VOLUME_PATH}/GB_YRFORECAST_databrisks.xlsx",   "2. Databricks File Path")
dbutils.widgets.text("sap_sheet_name",         "Sheet2",                                         "3. SAP Sheet Name")
dbutils.widgets.text("databricks_sheet_name",  "result",                                         "4. Databricks Sheet Name")
dbutils.widgets.text("stream_name",            "yrforecastn_dc02",                               "5. Stream Name")
dbutils.widgets.text("sap_table_name",         "YRFORECAST",                                     "6. SAP Table Name")
dbutils.widgets.text("primary_key_columns",    "0CALDAY,0DOC_NUMBER,0COMP_CODE", "7. Primary Key Columns")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  READ WIDGET VALUES + HARDCODED DEFAULTS
# ══════════════════════════════════════════════════════════════════

# Hardcoded — no widget needed
METADATA_DB       = "metadata_db"
RESULTS_DB        = "results_db"
NUMERIC_PRECISION = 2
EXCLUDE_COLUMNS   = []

# From widgets
source_file       = dbutils.widgets.get("sap_file_path").strip()
target_file       = dbutils.widgets.get("databricks_file_path").strip()
source_sheet      = dbutils.widgets.get("sap_sheet_name").strip()
target_sheet      = dbutils.widgets.get("databricks_sheet_name").strip()
stream_name       = dbutils.widgets.get("stream_name").strip()
sap_table         = dbutils.widgets.get("sap_table_name").strip()

pk_raw = dbutils.widgets.get("primary_key_columns").strip()
primary_key_columns = [c.strip() for c in pk_raw.split(",") if c.strip()]

print("=" * 70)
print("  CONFIGURATION")
print("=" * 70)
print(f"  Metadata DB       : {METADATA_DB}")
print(f"  Results DB        : {RESULTS_DB}")
print(f"  Stream            : {stream_name}")
print(f"  SAP Table         : {sap_table}")
print(f"  SAP File          : {source_file} [{source_sheet}]")
print(f"  Databricks File   : {target_file} [{target_sheet}]")
print(f"  PK Columns  ({len(primary_key_columns):>2d})  : {primary_key_columns}")
print(f"  Numeric Precision : {NUMERIC_PRECISION}")
print("=" * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CREATE DATABASES
# ══════════════════════════════════════════════════════════════════

spark.sql(f"CREATE DATABASE IF NOT EXISTS {METADATA_DB}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {RESULTS_DB}")
print(f"  Databases: {METADATA_DB}, {RESULTS_DB}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLE 1: Validation File Registry
# ══════════════════════════════════════════════════════════════════

spark.sql(f"DROP TABLE IF EXISTS {METADATA_DB}.validation_file_registry")
spark.sql(f"""
CREATE TABLE {METADATA_DB}.validation_file_registry (
    stream_name          STRING   COMMENT 'Unique identifier for this file pair',
    source_file_path     STRING   COMMENT 'Volume/DBFS path to SAP source Excel',
    source_sheet         STRING   COMMENT 'Sheet name in SAP source file',
    target_file_path     STRING   COMMENT 'Volume/DBFS path to Databricks target Excel',
    target_sheet         STRING   COMMENT 'Sheet name in Databricks target file',
    sap_table_name       STRING   COMMENT 'SAP table name',
    primary_key_columns  STRING   COMMENT 'Comma-separated PK column names',
    exclude_columns      STRING   COMMENT 'Comma-separated exclude columns for MINUS',
    numeric_precision    INT      COMMENT 'Decimal places for numeric comparison',
    is_active            STRING   COMMENT 'Y=active, N=disabled'
) USING DELTA
""")

spark.sql(f"""
INSERT INTO {METADATA_DB}.validation_file_registry VALUES (
    '{stream_name}', '{source_file}', '{source_sheet}',
    '{target_file}', '{target_sheet}', '{sap_table}',
    '{ ",".join(primary_key_columns) }', '{ ",".join(EXCLUDE_COLUMNS) }',
    {NUMERIC_PRECISION}, 'Y'
)
""")
print(f"  {METADATA_DB}.validation_file_registry — '{stream_name}' registered")
display(spark.table(f"{METADATA_DB}.validation_file_registry"))

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLE 2: Dynamic Column Mapping
# ══════════════════════════════════════════════════════════════════

spark.sql(f"DROP TABLE IF EXISTS {METADATA_DB}.dynamic_column_mapping")
spark.sql(f"""
CREATE TABLE {METADATA_DB}.dynamic_column_mapping (
    stream_name           STRING,
    source_column_name    STRING,
    source_column_index   INT,
    target_column_name    STRING,
    target_column_index   INT,
    mapping_method        STRING  COMMENT 'EXACT | NORMALIZED | FUZZY(score)',
    source_dtype          STRING,
    target_dtype          STRING,
    sap_field_name        STRING,
    sap_datatype          STRING,
    is_mapped             STRING,
    is_active             STRING
) USING DELTA
""")
print(f"  {METADATA_DB}.dynamic_column_mapping")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLE 3: SAP Source Schemas
# ══════════════════════════════════════════════════════════════════

spark.sql(f"DROP TABLE IF EXISTS {METADATA_DB}.sap_source_schemas")
spark.sql(f"""
CREATE TABLE {METADATA_DB}.sap_source_schemas (
    sap_table       STRING,
    field_name      STRING,
    data_element    STRING,
    data_type       STRING,
    field_length    STRING,
    description     STRING
) USING DELTA
""")
print(f"  {METADATA_DB}.sap_source_schemas")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLE 4: Validation Summary (all 13 checks)
# ══════════════════════════════════════════════════════════════════

spark.sql(f"DROP TABLE IF EXISTS {RESULTS_DB}.src_tgt_validation_summary")
spark.sql(f"""
CREATE TABLE {RESULTS_DB}.src_tgt_validation_summary (
    run_id          STRING, stream_name    STRING,
    source_file     STRING, target_file    STRING,
    check_name      STRING, check_category STRING,
    status          STRING, details         STRING,
    source_value    STRING, target_value   STRING,
    created_ts      TIMESTAMP
) USING DELTA
""")
print(f"  {RESULTS_DB}.src_tgt_validation_summary")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLE 5: Column-Level Detail
# ══════════════════════════════════════════════════════════════════

spark.sql(f"DROP TABLE IF EXISTS {RESULTS_DB}.src_tgt_column_validation")
spark.sql(f"""
CREATE TABLE {RESULTS_DB}.src_tgt_column_validation (
    run_id         STRING, stream_name   STRING,
    source_column  STRING, target_column STRING,
    check_name     STRING, status         STRING,
    source_value   STRING, target_value   STRING,
    difference     STRING, created_ts     TIMESTAMP
) USING DELTA
""")
print(f"  {RESULTS_DB}.src_tgt_column_validation")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLE 6: PK-Based Key Mismatches (source_minus_target / target_minus_source)
# ══════════════════════════════════════════════════════════════════

spark.sql(f"DROP TABLE IF EXISTS {RESULTS_DB}.src_tgt_key_mismatches")
spark.sql(f"""
CREATE TABLE {RESULTS_DB}.src_tgt_key_mismatches (
    run_id              STRING, stream_name        STRING,
    check_type          STRING, primary_key_values STRING,
    column_name         STRING, source_value       STRING,
    target_value        STRING, details            STRING,
    created_ts          TIMESTAMP
) USING DELTA
""")
print(f"  {RESULTS_DB}.src_tgt_key_mismatches")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLE 7: PK Issue Summary
# ══════════════════════════════════════════════════════════════════

spark.sql(f"DROP TABLE IF EXISTS {RESULTS_DB}.src_tgt_pk_issue_summary")
spark.sql(f"""
CREATE TABLE {RESULTS_DB}.src_tgt_pk_issue_summary (
    run_id             STRING,  stream_name        STRING,
    issue_type         STRING,  primary_key_values STRING,
    mismatched_columns STRING,  details            STRING,
    created_ts         TIMESTAMP
) USING DELTA
""")
print(f"  {RESULTS_DB}.src_tgt_pk_issue_summary")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLE 8: Exclude Column Audit (MINUS query)
# ══════════════════════════════════════════════════════════════════

spark.sql(f"DROP TABLE IF EXISTS {RESULTS_DB}.src_tgt_excluded_columns")
spark.sql(f"""
CREATE TABLE {RESULTS_DB}.src_tgt_excluded_columns (
    run_id          STRING, stream_name      STRING,
    column_name     STRING, exclusion_source STRING,
    reason          STRING, created_ts       TIMESTAMP
) USING DELTA
""")
print(f"  {RESULTS_DB}.src_tgt_excluded_columns")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLE 9: MINUS Query Results
# ══════════════════════════════════════════════════════════════════

spark.sql(f"DROP TABLE IF EXISTS {RESULTS_DB}.src_tgt_minus_results")
spark.sql(f"""
CREATE TABLE {RESULTS_DB}.src_tgt_minus_results (
    run_id      STRING, stream_name STRING,
    direction   STRING, row_data    STRING,
    created_ts  TIMESTAMP
) USING DELTA
""")
print(f"  {RESULTS_DB}.src_tgt_minus_results")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  VERIFY SETUP
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  SETUP COMPLETE")
print("=" * 70)

for db in [METADATA_DB, RESULTS_DB]:
    tables = [t['tableName'] for t in spark.sql(f"SHOW TABLES IN {db}").collect()]
    print(f"\n  {db} ({len(tables)} tables):")
    for t in sorted(tables):
        print(f"    {t}")

print(f"\n  Stream '{stream_name}' registered with {len(primary_key_columns)} PK columns")
print("=" * 70)
print(f"\n  NEXT -> Run 02_dynamic_column_mapper (stream_name={stream_name})")